# =============================================================================
# FACIAL EMOTION RECOGNITION - TRANSFER LEARNING EXPERIMENT TEMPLATE
# =============================================================================
# 
# Universal template for Transfer Learning experiments
# Supports VGG16, ResNet50, EfficientNet, DenseNet with layer unfreezing
# Change parameters in Cell 1 to run different experiments
# 
# Author: Pavlo Borysov
# Date: 12/10/2025
# =============================================================================


In [ ]:
# =============================================================================
# TRANSFER LEARNING EXPERIMENT CONFIGURATION - CHANGE ONLY HERE!
# =============================================================================

# Experiment identification
EXPERIMENT_NAME = "tl_test_experiment"
EXPERIMENT_DESCRIPTION = "Testing TL template"

# Base model selection
BASE_MODEL = "DenseNet121"  # Options: "VGG16", "ResNet50", "EfficientNetB2", "DenseNet121"

# Layer unfreezing strategy
UNFREEZE_STRATEGY = "last_blocks"  # Options: "none", "last_blocks", "custom_layers", "all"
UNFREEZE_LAST_N_BLOCKS = 2  # Number of blocks to unfreeze (if strategy = "last_blocks")
CUSTOM_UNFREEZE_LAYERS = []  # List of layer names to unfreeze (if strategy = "custom_layers")

# Training parameters
LEARNING_RATE_BASE = 0.001  # Learning rate for base model (if unfrozen)
LEARNING_RATE_HEAD = 0.01   # Learning rate for head layers
EPOCHS_FROZEN = 10          # Epochs with frozen base
EPOCHS_UNFROZEN = 20        # Epochs with unfrozen base
BATCH_SIZE = 32             # Smaller batch size for TL
PATIENCE = 5                # Early stopping patience

# Image parameters (TL models often need larger images)
IMG_SIZE = (224, 224)       # Standard size for most TL models
COLOR_MODE = "rgb"          # Always RGB for TL models
AUGMENT = True

# Head architecture
HEAD_DROPOUT = 0.5          # Dropout in head layers
HEAD_DENSE_UNITS = 512      # Dense layer units in head

# Auto-generated names
MODEL_NAME = f"{BASE_MODEL.lower()}_{EXPERIMENT_NAME}"
RUN_DIR_NAME = f"tl_{BASE_MODEL.lower()}_{EXPERIMENT_NAME}"

# Display configuration
print("=" * 80)
print(f"TRANSFER LEARNING EXPERIMENT: {EXPERIMENT_NAME}")
print("=" * 80)
print(f"Description: {EXPERIMENT_DESCRIPTION}")
print(f"Base model: {BASE_MODEL}")
print(f"Unfreeze strategy: {UNFREEZE_STRATEGY}")
if UNFREEZE_STRATEGY == "last_blocks":
    print(f"Unfreeze last {UNFREEZE_LAST_N_BLOCKS} blocks")
elif UNFREEZE_STRATEGY == "custom_layers":
    print(f"Custom layers to unfreeze: {CUSTOM_UNFREEZE_LAYERS}")
print(f"Image size: {IMG_SIZE}")
print(f"Learning rates: base={LEARNING_RATE_BASE}, head={LEARNING_RATE_HEAD}")
print(f"Training phases: frozen={EPOCHS_FROZEN}, unfrozen={EPOCHS_UNFROZEN}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Head dropout: {HEAD_DROPOUT}")
print(f"Model name: {MODEL_NAME}")
print("=" * 80)


In [ ]:
# =============================================================================
# IMPORTS AND SETUP
# =============================================================================

import warnings
import os
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import time
import json
from pathlib import Path

from sklearn.metrics import classification_report, confusion_matrix, f1_score

import tensorflow as tf
import keras
from keras import layers, applications
from keras.utils import image_dataset_from_directory
from keras.optimizers import Adam

# Fixed random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")
print(f"Random seed: {RANDOM_SEED}")


In [ ]:
# =============================================================================
# CONSTANTS AND PATHS
# =============================================================================

# Data paths
DATA_DIR = Path("../../data")
TRAIN_DATA_DIR = Path("../../data/train")
VALIDATION_DATA_DIR = Path("../../data/validation")
TEST_DATA_DIR = Path("../../data/test")

# Output paths
MODELS_DIR = Path("../../models")
RUNS_DIR = Path("../../runs")
REPORTS_DIR = Path("../../reports")

# Fixed class order and weights (from main notebook)
CLASS_ORDER = ['happy', 'neutral', 'sad', 'surprise']
CLASS_WEIGHTS = {
    0: 0.950,  # happy
    1: 0.950,  # neutral  
    2: 0.949,  # sad
    3: 1.190   # surprise
}

# Model parameters
NUM_CLASSES = 4
METRIC_NAME = "sparse_categorical_accuracy"
VAL_METRIC_NAME = f"val_{METRIC_NAME}"

print("Setup complete!")
print(f"Class order: {CLASS_ORDER}")
print(f"Class weights: {CLASS_WEIGHTS}")


In [ ]:
# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def save_metrics(metrics: dict, out_json: Path):
    """Save metrics to JSON file."""
    with open(out_json, "w") as f:
        json.dump(metrics, f, indent=2)

def make_run_dir(name: str) -> Path:
    """Create unique directory for experiment."""
    ts = datetime.now().strftime("%Y%m%d-%H%M%S")
    run_dir = RUNS_DIR / f"{ts}_{name}"
    (run_dir / "figs").mkdir(parents=True, exist_ok=True)
    return run_dir

def print_dict_table(d: dict, title="Dictionary", val_header="Value", show_percent=False, value_decimals=3):
    """Print dictionary as formatted table."""
    print(f"\n{title}")
    print(f"{'Class':<8} {val_header}")
    print("-" * (8 + len(val_header) + 2))
    total = sum(d.values()) if show_percent else None
    
    for key, value in d.items():
        if show_percent and total:
            pct = f" ({value/total*100:.1f}%)"
            print(f"{key:<8} {value:.{value_decimals}f}{pct}")
        else:
            print(f"{key:<8} {value:.{value_decimals}f}")
    
    if show_percent and total:
        print("-" * (8 + len(val_header) + 2))
        print(f"{'TOTAL':<8} {total:.{value_decimals}f}")
    print()

print("Utility functions loaded!")


In [ ]:
# =============================================================================
# DATA LOADERS
# =============================================================================

def make_generators(train_dir, val_dir, test_dir, img_size=(224,224),
                    color_mode="rgb", batch_size=32, augment=True):
    """Create data generators for TL models."""
    
    def preprocess_image(image, label):
        image = tf.cast(image, tf.float32) / 255.0
        return image, label
    
    def augment_image_tl(image, label):
        # TL-specific augmentation
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_brightness(image, 0.3)
        image = tf.image.random_contrast(image, 0.8, 1.2)
        image = tf.image.random_saturation(image, 0.8, 1.2)
        image = tf.clip_by_value(image, 0.0, 1.0)
        return image, label
    
    print("Creating TL data loaders...")
    
    # Train data
    train_ds = image_dataset_from_directory(
        train_dir,
        class_names=CLASS_ORDER,
        image_size=img_size,
        batch_size=batch_size,
        color_mode=color_mode,
        shuffle=True
    )
    train_ds = train_ds.map(preprocess_image)
    if augment:
        train_ds = train_ds.map(augment_image_tl)
    
    # Validation data
    val_ds = image_dataset_from_directory(
        val_dir,
        class_names=CLASS_ORDER,
        image_size=img_size,
        batch_size=batch_size,
        color_mode=color_mode,
        shuffle=False
    )
    val_ds = val_ds.map(preprocess_image)
    
    # Test data
    test_ds = image_dataset_from_directory(
        test_dir,
        class_names=CLASS_ORDER,
        image_size=img_size,
        batch_size=batch_size,
        color_mode=color_mode,
        shuffle=False
    )
    test_ds = test_ds.map(preprocess_image)
    
    return train_ds, val_ds, test_ds

# Create data loaders
train_ds, val_ds, test_ds = make_generators(
    train_dir=TRAIN_DATA_DIR,
    val_dir=VALIDATION_DATA_DIR,
    test_dir=TEST_DATA_DIR,
    img_size=IMG_SIZE,
    color_mode=COLOR_MODE,
    batch_size=BATCH_SIZE,
    augment=AUGMENT
)

# Data sanity check
for xb, yb in train_ds.take(1):
    break

print(f"\nTL Data loaded successfully!")
print(f"Train batch shape: {xb.shape}")
print(f"Labels shape: {yb.shape}")
print(f"Image range: [{tf.reduce_min(xb):.3f}, {tf.reduce_max(xb):.3f}]")
print(f"Unique labels: {tf.unique(yb)[0].numpy()}")
print(f"Augmentation: {AUGMENT}")


In [ ]:
# =============================================================================
# TRANSFER LEARNING MODEL BUILDER
# =============================================================================

def get_base_model(model_name, input_shape):
    """Get base model for transfer learning."""
    if model_name == "VGG16":
        base_model = applications.VGG16(
            weights='imagenet',
            include_top=False,
            input_shape=input_shape
        )
    elif model_name == "ResNet50":
        base_model = applications.ResNet50V2(
            weights='imagenet',
            include_top=False,
            input_shape=input_shape
        )
    elif model_name == "EfficientNetB2":
        base_model = applications.EfficientNetB2(
            weights='imagenet',
            include_top=False,
            input_shape=input_shape
        )
    elif model_name == "DenseNet121":
        base_model = applications.DenseNet121(
            weights='imagenet',
            include_top=False,
            input_shape=input_shape
        )
    else:
        raise ValueError(f"Unknown model: {model_name}")
    
    return base_model

def unfreeze_layers(base_model, strategy, last_n_blocks=None, custom_layers=None):
    """Unfreeze layers based on strategy."""
    # First freeze all layers
    base_model.trainable = False
    
    if strategy == "none":
        print("No layers unfrozen - using frozen base model")
        return
    
    elif strategy == "all":
        base_model.trainable = True
        print("All layers unfrozen")
        return
    
    elif strategy == "last_blocks" and last_n_blocks:
        # Unfreeze last N blocks
        base_model.trainable = True
        total_layers = len(base_model.layers)
        freeze_until = total_layers - last_n_blocks * 10  # Approximate blocks
        
        for layer in base_model.layers[:freeze_until]:
            layer.trainable = False
        
        print(f"Unfrozen last {last_n_blocks} blocks (layers {freeze_until}-{total_layers})")
        return
    
    elif strategy == "custom_layers" and custom_layers:
        base_model.trainable = True
        for layer in base_model.layers:
            if layer.name in custom_layers:
                layer.trainable = True
            else:
                layer.trainable = False
        print(f"Unfrozen custom layers: {custom_layers}")
        return
    
    else:
        raise ValueError(f"Invalid unfreeze strategy: {strategy}")

def build_tl_model(base_model_name, input_shape, num_classes, 
                   head_dropout=0.5, head_dense_units=512):
    """Build complete TL model with custom head."""
    
    # Get base model
    base_model = get_base_model(base_model_name, input_shape)
    base_model.trainable = False
    
    # Build head
    inputs = keras.Input(shape=input_shape, name="input")
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
    x = layers.Dropout(head_dropout, name="dropout_head")(x)
    x = layers.Dense(head_dense_units, activation="relu", name="dense_head")(x)
    x = layers.Dropout(head_dropout, name="dropout_final")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="output")(x)
    
    # Create model
    model = keras.Model(inputs, outputs, name=f"tl_{base_model_name.lower()}")
    
    return model, base_model

print("TL model builder loaded!")


In [ ]:
# =============================================================================
# MODEL CREATION AND SETUP
# =============================================================================

# Create run directory
run_dir = make_run_dir(RUN_DIR_NAME)
print(f"Run directory: {run_dir}")

# Input shape
input_shape = IMG_SIZE + (3,)  # Always 3 channels for TL models

# Build model
print(f"\nBuilding {BASE_MODEL} model...")
model, base_model = build_tl_model(
    base_model_name=BASE_MODEL,
    input_shape=input_shape,
    num_classes=NUM_CLASSES,
    head_dropout=HEAD_DROPOUT,
    head_dense_units=HEAD_DENSE_UNITS
)

# Initial compilation (frozen base)
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE_HEAD),
    loss="sparse_categorical_crossentropy",
    metrics=[METRIC_NAME]
)

# Display model info
print(f"\nModel created: {MODEL_NAME}")
print(f"Input shape: {input_shape}")
print(f"Total parameters: {model.count_params():,}")
print(f"Trainable parameters: {sum([tf.keras.backend.count_params(w) for w in model.trainable_weights]):,}")

# Save model summary
with open(run_dir / "model_summary.txt", "w") as f:
    model.summary(print_fn=lambda x: f.write(x + '\n'))

print(f"Model summary saved to: {run_dir / 'model_summary.txt'}")


In [ ]:
# =============================================================================
# TRAINING PHASE 1: FROZEN BASE MODEL
# =============================================================================

print("=" * 70)
print("PHASE 1: TRAINING WITH FROZEN BASE MODEL")
print("=" * 70)

# Callbacks for frozen training
callbacks_frozen = [
    keras.callbacks.EarlyStopping(
        monitor=VAL_METRIC_NAME,
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor=VAL_METRIC_NAME,
        factor=0.5,
        patience=PATIENCE//2,
        min_lr=1e-7,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=run_dir / f"{MODEL_NAME}_frozen.keras",
        monitor=VAL_METRIC_NAME,
        save_best_only=True,
        verbose=1
    )
]

# Training configuration
print(f"Epochs: {EPOCHS_FROZEN}")
print(f"Learning rate: {LEARNING_RATE_HEAD}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Class weights: {CLASS_WEIGHTS}")
print("=" * 70)

# Start training
start_time = time.time()
print(f"Training started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

history_frozen = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FROZEN,
    callbacks=callbacks_frozen,
    class_weight=CLASS_WEIGHTS,
    verbose=1
)

frozen_time = time.time() - start_time
print(f"\nFrozen training completed in {frozen_time/60:.1f} minutes")
print(f"Best validation accuracy: {max(history_frozen.history[VAL_METRIC_NAME]):.4f}")


In [ ]:
# =============================================================================
# TRAINING PHASE 2: UNFROZEN BASE MODEL (if applicable)
# =============================================================================

if UNFREEZE_STRATEGY != "none":
    print("=" * 70)
    print("PHASE 2: UNFREEZING AND FINE-TUNING BASE MODEL")
    print("=" * 70)
    
    # Unfreeze layers
    unfreeze_layers(
        base_model, 
        strategy=UNFREEZE_STRATEGY,
        last_n_blocks=UNFREEZE_LAST_N_BLOCKS,
        custom_layers=CUSTOM_UNFREEZE_LAYERS
    )
    
    # Recompile with lower learning rate for base model
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE_BASE),
        loss="sparse_categorical_crossentropy",
        metrics=[METRIC_NAME]
    )
    
    # Callbacks for unfrozen training
    callbacks_unfrozen = [
        keras.callbacks.EarlyStopping(
            monitor=VAL_METRIC_NAME,
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor=VAL_METRIC_NAME,
            factor=0.5,
            patience=PATIENCE//2,
            min_lr=1e-7,
            verbose=1
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=run_dir / f"{MODEL_NAME}_unfrozen.keras",
            monitor=VAL_METRIC_NAME,
            save_best_only=True,
            verbose=1
        )
    ]
    
    # Training configuration
    print(f"Epochs: {EPOCHS_UNFROZEN}")
    print(f"Learning rate: {LEARNING_RATE_BASE}")
    print(f"Batch size: {BATCH_SIZE}")
    print(f"Class weights: {CLASS_WEIGHTS}")
    print("=" * 70)
    
    # Start unfrozen training
    start_time = time.time()
    print(f"Unfrozen training started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    history_unfrozen = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_UNFROZEN,
        callbacks=callbacks_unfrozen,
        class_weight=CLASS_WEIGHTS,
        verbose=1
    )
    
    unfrozen_time = time.time() - start_time
    print(f"\nUnfrozen training completed in {unfrozen_time/60:.1f} minutes")
    print(f"Best validation accuracy: {max(history_unfrozen.history[VAL_METRIC_NAME]):.4f}")
    
    # Combine histories
    combined_history = {
        'loss': history_frozen.history['loss'] + history_unfrozen.history['loss'],
        'val_loss': history_frozen.history['val_loss'] + history_unfrozen.history['val_loss'],
        METRIC_NAME: history_frozen.history[METRIC_NAME] + history_unfrozen.history[METRIC_NAME],
        VAL_METRIC_NAME: history_frozen.history[VAL_METRIC_NAME] + history_unfrozen.history[VAL_METRIC_NAME]
    }
    total_training_time = frozen_time + unfrozen_time
    
else:
    print("Skipping Phase 2 - no unfreezing strategy specified")
    combined_history = history_frozen.history
    total_training_time = frozen_time

print(f"\nTotal training time: {total_training_time/60:.1f} minutes")


In [ ]:
# =============================================================================
# EVALUATION AND RESULTS
# =============================================================================

print("=" * 70)
print(f"EVALUATION - {MODEL_NAME}")
print("=" * 70)

# Get predictions
y_true = []
y_pred = []

for batch in test_ds:
    images, labels = batch
    predictions = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

# Calculate metrics
test_accuracy = f1_score(y_true, y_pred, average='weighted')
test_macro_f1 = f1_score(y_true, y_pred, average='macro')
test_weighted_f1 = f1_score(y_true, y_pred, average='weighted')

# Display results
print(f"Test accuracy: {test_accuracy:.3f}")
print(f"Test macro F1: {test_macro_f1:.3f}")
print(f"Test weighted F1: {test_weighted_f1:.3f}")
print("=" * 70)

# Save final model
model.save(MODELS_DIR / f"best_{MODEL_NAME}.keras")
print(f"Final model saved to: {MODELS_DIR / f'best_{MODEL_NAME}.keras'}")

# Save results
results = {
    "experiment_name": EXPERIMENT_NAME,
    "experiment_description": EXPERIMENT_DESCRIPTION,
    "model_name": MODEL_NAME,
    "base_model": BASE_MODEL,
    "unfreeze_strategy": UNFREEZE_STRATEGY,
    "unfreeze_last_n_blocks": UNFREEZE_LAST_N_BLOCKS,
    "custom_unfreeze_layers": CUSTOM_UNFREEZE_LAYERS,
    "input_shape": list(input_shape),
    "head_dropout": HEAD_DROPOUT,
    "head_dense_units": HEAD_DENSE_UNITS,
    "learning_rate_base": LEARNING_RATE_BASE,
    "learning_rate_head": LEARNING_RATE_HEAD,
    "epochs_frozen": EPOCHS_FROZEN,
    "epochs_unfrozen": EPOCHS_UNFROZEN,
    "batch_size": BATCH_SIZE,
    "augment": AUGMENT,
    "training_time_minutes": total_training_time / 60,
    "test_accuracy": test_accuracy,
    "test_macro_f1": test_macro_f1,
    "test_weighted_f1": test_weighted_f1,
    "total_epochs_trained": len(combined_history['loss']),
    "final_train_acc": combined_history[METRIC_NAME][-1],
    "final_train_loss": combined_history['loss'][-1],
    "final_val_acc": combined_history[VAL_METRIC_NAME][-1],
    "final_val_loss": combined_history['val_loss'][-1],
    "best_val_acc": max(combined_history[VAL_METRIC_NAME]),
    "best_epoch": combined_history[VAL_METRIC_NAME].index(max(combined_history[VAL_METRIC_NAME])) + 1
}

# Save to run directory
save_metrics(results, run_dir / "experiment_results.json")
save_metrics(results, run_dir / "config.json")

print(f"\nResults saved to: {run_dir}")
print(f"Experiment completed: {EXPERIMENT_NAME}")


In [ ]:
# =============================================================================
# VISUALIZATION
# =============================================================================

# Training history plots
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy plot
axes[0].plot(combined_history[METRIC_NAME], label='Train', linewidth=2)
axes[0].plot(combined_history[VAL_METRIC_NAME], label='Validation', linewidth=2)
axes[0].set_title(f'Accuracy - {MODEL_NAME}')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(combined_history['loss'], label='Train', linewidth=2)
axes[1].plot(combined_history['val_loss'], label='Validation', linewidth=2)
axes[1].set_title(f'Loss - {MODEL_NAME}')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(run_dir / "training_history.png", dpi=150, bbox_inches="tight")
plt.show()

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar()

tick_marks = np.arange(len(CLASS_ORDER))
plt.xticks(tick_marks, CLASS_ORDER, rotation=45)
plt.yticks(tick_marks, CLASS_ORDER)

# Add text annotations
thresh = cm.max() / 2.
for i, j in np.ndindex(cm.shape):
    plt.text(j, i, cm[i, j], horizontalalignment="center",
             color="white" if cm[i, j] > thresh else "black", fontsize=10)

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {MODEL_NAME}')
plt.tight_layout()
plt.savefig(run_dir / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Visualizations saved to: {run_dir}")
print("=" * 70)
print(f"✓ {MODEL_NAME} Training and Evaluation Complete!")
print("=" * 70)


# =============================================================================
# EXPERIMENT COMPLETED!
# =============================================================================

## 📊 **Results Summary:**
- **Model**: {MODEL_NAME}
- **Base Model**: {BASE_MODEL}
- **Unfreeze Strategy**: {UNFREEZE_STRATEGY}
- **Test Accuracy**: {test_accuracy:.3f}

## 🎯 **Next Steps:**
1. **Copy this notebook** for new experiments
2. **Change parameters** in Cell 1 (Configuration)
3. **Run experiment** and compare results
4. **Try different models**: VGG16, ResNet50, EfficientNetB2, DenseNet121
5. **Experiment with unfreezing**: none, last_blocks, custom_layers, all

## 🔧 **Configuration Tips:**
- **Start with frozen base** (UNFREEZE_STRATEGY = "none")
- **Then try last 2 blocks** (UNFREEZE_STRATEGY = "last_blocks", UNFREEZE_LAST_N_BLOCKS = 2)
- **Adjust learning rates** based on results
- **Use smaller batch sizes** for larger models

## 📁 **Files Generated:**
- Model: `models/best_{MODEL_NAME}.keras`
- Results: `runs/{timestamp}_{RUN_DIR_NAME}/`
- Visualizations: Training history, confusion matrix
